In [1]:
import os
import pandas as pd

print(bool(os.getenv("GENAI_API_KEY")))

True


In [2]:
# ============================================
# STEP 4: INITIALIZE MAAS LLM
# ============================================

from langchain_openai import ChatOpenAI
import httpx
import os

api_key = os.getenv("GENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GENAI_API_KEY is not available. "
        "Activate/set the MaaS API key before running this notebook."
    )

client = httpx.Client(verify=False)

llm = ChatOpenAI(
    base_url="https://genailab.tcs.in",
    model="genailab-maas-gpt-35-turbo",
    api_key=api_key,
    http_client=client
)

print("MaaS LLM initialized successfully.")

MaaS LLM initialized successfully.


In [3]:
# ============================================
# STEP 5: TEST MAAS LLM
# ============================================

response = llm.invoke(
    "Hi. Reply with exactly: LLM connection successful."
)

print(response.content)

LLM connection successful.


In [4]:
# ============================================
# STEP 6: LOAD ORCHESTRATOR EVENTS
# ============================================

events_path = "../data/processed/orchestrated_runtime_events.csv"

events_df = pd.read_csv(events_path)

print("Orchestrator events loaded.")
print("Shape:", events_df.shape)

display(events_df.head())

Orchestrator events loaded.
Shape: (1930, 15)


,Cycle,Machine_ID,Machine_Type,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,agents_invoked,trigger_reason,maintenance_priority,maintenance_decision,alert_level,alert_required,alert_message,maintenance_reason
0,1,CNC_001,CNC_Mill,BEARING_WEAR,0.0001,256.45,HEALTHY,False,NORMAL,NONE,NO ACTION,NONE,False,NaN,NaN
1,1,COMP_003,Compressor,VIBRATION,0.0001,284.57,HEALTHY,False,NORMAL,NONE,NO ACTION,NONE,False,NaN,NaN
2,1,MOTOR_004,Electric_Motor,ELECTRICAL,0.0001,238.10,HEALTHY,False,NORMAL,NONE,NO ACTION,NONE,False,NaN,NaN
3,1,PRESS_006,Hydraulic_Press,PRESSURE,0.0001,273.77,HEALTHY,False,NORMAL,NONE,NO ACTION,NONE,False,NaN,NaN
4,1,PUMP_002,Hydraulic_Pump,OVERHEATING,0.0001,306.94,HEALTHY,False,NORMAL,NONE,NO ACTION,NONE,False,NaN,NaN


In [5]:
# ============================================
# STEP 7: SELECT ACTIONABLE EVENTS
# ============================================

actionable_events = events_df[
    events_df["agents_invoked"] == True
].copy()

print("Total runtime events:", len(events_df))
print("Actionable events:", len(actionable_events))

display(actionable_events.head())

Total runtime events: 1930
Actionable events: 55


,Cycle,Machine_ID,Machine_Type,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,agents_invoked,trigger_reason,maintenance_priority,maintenance_decision,alert_level,alert_required,alert_message,maintenance_reason
1097,157,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0002,49.12,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1125,161,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0003,48.37,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1139,163,ROBOT_007,Industrial_Robot,ACTUATOR_DEGRADATION,0.0002,48.41,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | ROBOT_007 (Industrial_Robot) | Cycle...,Machine degradation requires inspection.
1206,173,MOTOR_004,Electric_Motor,ELECTRICAL,0.0006,50.00,HEALTHY,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | MOTOR_004 (Electric_Motor) | Cycle 1...,Machine degradation requires inspection.
1241,178,MOTOR_004,Electric_Motor,ELECTRICAL,0.0006,48.63,WARNING,True,STATE_CHANGE,HIGH,SCHEDULE INSPECTION,WARNING,True,WARNING | MOTOR_004 (Electric_Motor) | Cycle 1...,Machine degradation requires inspection.


In [6]:
print(actionable_events.columns.tolist())

['Cycle', 'Machine_ID', 'Machine_Type', 'Simulation_Fault', 'failure_probability', 'predicted_rul_cycles', 'overall_status', 'agents_invoked', 'trigger_reason', 'maintenance_priority', 'maintenance_decision', 'alert_level', 'alert_required', 'alert_message', 'maintenance_reason']


In [7]:
# ============================================
# STEP 9: BUILD MACHINE CONTEXT
# ============================================

def build_machine_context(row):

    return {
        "machine_id": row["Machine_ID"],
        "machine_type": row["Machine_Type"],
        "cycle": int(row["Cycle"]),

        "simulation_fault": row["Simulation_Fault"],

        "failure_probability": float(
            row["failure_probability"]
        ),

        "predicted_rul_cycles": float(
            row["predicted_rul_cycles"]
        ),

        "overall_status": row["overall_status"],

        "trigger_reason": row["trigger_reason"],

        "maintenance_priority": row[
            "maintenance_priority"
        ],

        "maintenance_decision": row[
            "maintenance_decision"
        ],

        "maintenance_reason": row[
            "maintenance_reason"
        ],

        "alert_level": row["alert_level"],

        "alert_required": bool(
            row["alert_required"]
        ),

        "alert_message": row["alert_message"]
    }

print("Machine context function created.")

Machine context function created.


In [8]:
# ============================================
# STEP 10: SELECT ONE ACTIONABLE EVENT
# ============================================

sample_event = actionable_events.iloc[0]

sample_context = build_machine_context(
    sample_event
)

import json

print(
    json.dumps(
        sample_context,
        indent=2
    )
)

{
  "machine_id": "ROBOT_007",
  "machine_type": "Industrial_Robot",
  "cycle": 157,
  "simulation_fault": "ACTUATOR_DEGRADATION",
  "failure_probability": 0.0002,
  "predicted_rul_cycles": 49.12,
  "overall_status": "WARNING",
  "trigger_reason": "STATE_CHANGE",
  "maintenance_priority": "HIGH",
  "maintenance_decision": "SCHEDULE INSPECTION",
  "maintenance_reason": "Machine degradation requires inspection.",
  "alert_level": "WARNING",
  "alert_required": true,
  "alert_message": "WARNING | ROBOT_007 (Industrial_Robot) | Cycle 157 | Failure probability=0.02% | RUL=49.12 cycles | Action=SCHEDULE INSPECTION"
}


In [9]:
# ============================================
# STEP 11: LLM SYSTEM PROMPT
# ============================================

SYSTEM_PROMPT = """
You are the Maintenance Intelligence Agent
for a smart manufacturing predictive
maintenance system.

Your input comes from a deterministic
predictive maintenance pipeline consisting of:

1. Failure Prediction Model
2. RUL Prediction Model
3. Smart Twin
4. Maintenance Agent
5. Alert Agent
6. Orchestrator

Your responsibility is to explain the
machine condition and the maintenance
decision to a human maintenance operator.

IMPORTANT RULES:

- Do NOT override the maintenance decision.
- Do NOT change the alert level.
- Do NOT invent sensor values.
- Do NOT invent machine measurements.
- Do NOT claim that a physical root cause
  is confirmed.
- If discussing a possible cause, clearly
  describe it as a hypothesis.
- Use only the information provided in
  the machine event.
- Distinguish between model prediction,
  possible cause, and confirmed information.
- Keep the explanation practical and concise.
- Follow the existing maintenance decision.

Your response must contain these sections:

1. ASSESSMENT
   Explain the current machine condition.

2. EVIDENCE
   Mention the important numerical and
   decision-related evidence.

3. POSSIBLE CONTRIBUTING FACTORS
   Discuss possible factors only as hypotheses
   based on the supplied information.

4. RECOMMENDED INSPECTION
   Explain what the maintenance team should
   inspect.

5. OPERATOR ACTION
   Explain what action should be taken,
   while preserving the existing maintenance
   decision.

Do not produce a generic explanation.
Your response must be specific to the
machine event provided.
"""

print("LLM system prompt created.")

LLM system prompt created.


In [10]:
# ============================================
# STEP 12: BUILD USER PROMPT
# ============================================

def build_user_prompt(machine_context):

    return f"""
Analyze the following real-time machine event
generated by our Smart Manufacturing system.

MACHINE EVENT:

{json.dumps(machine_context, indent=2)}

Provide a maintenance-oriented explanation
using the required sections from your system
instructions.

Remember:

- Use only the supplied information.
- Do not invent measurements.
- Do not override the maintenance decision.
- Clearly distinguish predictions from hypotheses.
"""

In [11]:
print("User prompt function created.")

User prompt function created.


In [12]:
# ============================================
# STEP 13: PREVIEW LLM PROMPT
# ============================================

user_prompt = build_user_prompt(
    sample_context
)

print(user_prompt)


Analyze the following real-time machine event
generated by our Smart Manufacturing system.

MACHINE EVENT:

{
  "machine_id": "ROBOT_007",
  "machine_type": "Industrial_Robot",
  "cycle": 157,
  "simulation_fault": "ACTUATOR_DEGRADATION",
  "failure_probability": 0.0002,
  "predicted_rul_cycles": 49.12,
  "overall_status": "WARNING",
  "trigger_reason": "STATE_CHANGE",
  "maintenance_priority": "HIGH",
  "maintenance_decision": "SCHEDULE INSPECTION",
  "maintenance_reason": "Machine degradation requires inspection.",
  "alert_level": "WARNING",
  "alert_required": true,
  "alert_message": "WARNING | ROBOT_007 (Industrial_Robot) | Cycle 157 | Failure probability=0.02% | RUL=49.12 cycles | Action=SCHEDULE INSPECTION"
}

Provide a maintenance-oriented explanation
using the required sections from your system
instructions.

Remember:

- Use only the supplied information.
- Do not invent measurements.
- Do not override the maintenance decision.
- Clearly distinguish predictions from hypothe

In [13]:
# ============================================
# STEP 14: LLM MAINTENANCE AGENT
# ============================================

def llm_maintenance_agent(machine_context):

    user_prompt = build_user_prompt(
        machine_context
    )

    response = llm.invoke(
        [
            (
                "system",
                SYSTEM_PROMPT
            ),
            (
                "human",
                user_prompt
            )
        ]
    )

    return response.content

print("LLM Maintenance Agent created.")

LLM Maintenance Agent created.


In [14]:
# ============================================
# STEP 15: ANALYZE ONE REAL MACHINE EVENT
# ============================================

llm_report = llm_maintenance_agent(
    sample_context
)

print("========================================")
print("LLM MAINTENANCE INTELLIGENCE REPORT")
print("========================================")
print(llm_report)

LLM MAINTENANCE INTELLIGENCE REPORT
1. ASSESSMENT  
The industrial robot ROBOT_007 is currently showing signs of degradation, as indicated by a change in its operating state. While the immediate failure probability is low (0.02%), the remaining useful life is predicted to be about 49 cycles, suggesting that mechanical or actuator performance may be declining.

2. EVIDENCE  
- Failure probability: 0.0002 (0.02%)—indicates a low immediate risk of failure.  
- Predicted remaining useful life (RUL): 49.12 cycles—indicates limited time before potential failure.  
- Overall status: WARNING with alert level WARNING.  
- Maintenance priority: HIGH and decision to SCHEDULE INSPECTION based on detected degradation.

3. POSSIBLE CONTRIBUTING FACTORS  
The simulated fault is actuator degradation, which may be causing subtle performance losses or inefficiencies. This is a hypothesis based on model simulation and may indicate wear or reduced responsiveness in actuator components affecting operation.

In [15]:
# ============================================
# STEP 16: GENERATE LLM REPORTS
# ============================================

llm_reports = []

for index, row in actionable_events.iterrows():

    try:

        context = build_machine_context(row)

        report = llm_maintenance_agent(
            context
        )

        llm_reports.append({
            "Machine_ID": row["Machine_ID"],
            "Machine_Type": row["Machine_Type"],
            "Cycle": row["Cycle"],
            "Simulation_Fault": row["Simulation_Fault"],
            "failure_probability": row["failure_probability"],
            "predicted_rul_cycles": row["predicted_rul_cycles"],
            "overall_status": row["overall_status"],
            "maintenance_priority": row["maintenance_priority"],
            "maintenance_decision": row["maintenance_decision"],
            "alert_level": row["alert_level"],
            "alert_required": row["alert_required"],
            "llm_report": report
        })

        print(
            f"Processed {len(llm_reports)}/{len(actionable_events)} "
            f"- {row['Machine_ID']} Cycle {row['Cycle']}"
        )

    except Exception as e:

        print(
            f"ERROR processing row {index}: {e}"
        )

Processed 1/55 - ROBOT_007 Cycle 157
Processed 2/55 - ROBOT_007 Cycle 161
Processed 3/55 - ROBOT_007 Cycle 163
Processed 4/55 - MOTOR_004 Cycle 173
Processed 5/55 - MOTOR_004 Cycle 178
Processed 6/55 - MOTOR_004 Cycle 180
Processed 7/55 - ROBOT_007 Cycle 189
Processed 8/55 - ROBOT_007 Cycle 190
Processed 9/55 - ROBOT_007 Cycle 191
Processed 10/55 - CNC_001 Cycle 193
Processed 11/55 - ROBOT_007 Cycle 193
Processed 12/55 - ROBOT_007 Cycle 194
Processed 13/55 - CNC_001 Cycle 195
Processed 14/55 - ROBOT_007 Cycle 195
Processed 15/55 - PRESS_006 Cycle 202
Processed 16/55 - PRESS_006 Cycle 206
Processed 17/55 - MOTOR_004 Cycle 208
Processed 18/55 - MOTOR_004 Cycle 210
Processed 19/55 - MOTOR_004 Cycle 211
Processed 20/55 - MOTOR_004 Cycle 212
Processed 21/55 - MOTOR_004 Cycle 216
Processed 22/55 - MOTOR_004 Cycle 217
Processed 23/55 - COMP_003 Cycle 218
Processed 24/55 - CNC_001 Cycle 221
Processed 25/55 - COMP_003 Cycle 222
Processed 26/55 - CNC_001 Cycle 223
Processed 27/55 - CNC_001 Cycle

# STEP 17 onward — Finalize LLM Maintenance Reports

The LLM batch-processing step above processes all actionable events produced by the orchestrator.

This section:
1. Converts the generated reports into a DataFrame.
2. Validates that reports are present and non-empty.
3. Displays a representative report.
4. Saves the reports to `data/processed/llm_maintenance_reports.csv`.
5. Reloads the saved file to verify persistence.
6. Produces summary statistics for the project/demo.

**Important:** Execute the notebook from the top after a fresh VM/venv restart. Do not skip the earlier cells because they initialize the MaaS connection, load the orchestrator events, define the prompts/agent, and generate `llm_reports`.


In [16]:
# ============================================
# STEP 17: CREATE LLM REPORT DATAFRAME
# ============================================

import pandas as pd

if "llm_reports" not in globals():
    raise RuntimeError(
        "llm_reports is not available. "
        "Execute the notebook from the beginning through STEP 16 first."
    )

llm_reports_df = pd.DataFrame(llm_reports)

print("LLM reports generated:", len(llm_reports_df))
print("Result shape:", llm_reports_df.shape)

display(llm_reports_df.head())


LLM reports generated: 55
Result shape: (55, 12)


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level,alert_required,llm_report
0,ROBOT_007,Industrial_Robot,157,ACTUATOR_DEGRADATION,0.0002,49.12,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
1,ROBOT_007,Industrial_Robot,161,ACTUATOR_DEGRADATION,0.0003,48.37,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
2,ROBOT_007,Industrial_Robot,163,ACTUATOR_DEGRADATION,0.0002,48.41,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
3,MOTOR_004,Electric_Motor,173,ELECTRICAL,0.0006,50.00,HEALTHY,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe electric motor MOTOR_004 ...
4,MOTOR_004,Electric_Motor,178,ELECTRICAL,0.0006,48.63,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,ASSESSMENT \nThe electric motor MOTOR_004 is ...


In [17]:
# ============================================
# STEP 18: VALIDATE LLM REPORTS
# ============================================

required_columns = [
    "Machine_ID",
    "Machine_Type",
    "Cycle",
    "Simulation_Fault",
    "failure_probability",
    "predicted_rul_cycles",
    "overall_status",
    "maintenance_priority",
    "maintenance_decision",
    "alert_level",
    "alert_required",
    "llm_report"
]

missing_columns = [
    col for col in required_columns
    if col not in llm_reports_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected columns: {missing_columns}"
    )

missing_reports = int(
    llm_reports_df["llm_report"].isna().sum()
)

empty_reports = int(
    llm_reports_df["llm_report"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("Missing reports:", missing_reports)
print("Empty reports:", empty_reports)

if missing_reports == 0 and empty_reports == 0:
    print("Validation successful: all LLM reports are present.")
else:
    print("WARNING: Some LLM reports are missing or empty.")


Missing reports: 0
Empty reports: 0
Validation successful: all LLM reports are present.


In [18]:
# ============================================
# STEP 19: DISPLAY SAMPLE LLM REPORT
# ============================================

sample_report = llm_reports_df.iloc[0]

print("============================================")
print("LLM MAINTENANCE INTELLIGENCE REPORT")
print("============================================")
print("Machine ID:", sample_report["Machine_ID"])
print("Machine Type:", sample_report["Machine_Type"])
print("Cycle:", sample_report["Cycle"])
print("Simulation Fault:", sample_report["Simulation_Fault"])
print("Overall Status:", sample_report["overall_status"])
print("Maintenance Priority:", sample_report["maintenance_priority"])
print("Maintenance Decision:", sample_report["maintenance_decision"])
print("Alert Level:", sample_report["alert_level"])
print("--------------------------------------------")
print(sample_report["llm_report"])


LLM MAINTENANCE INTELLIGENCE REPORT
Machine ID: ROBOT_007
Machine Type: Industrial_Robot
Cycle: 157
Simulation Fault: ACTUATOR_DEGRADATION
Overall Status: WARNING
Maintenance Priority: HIGH
Maintenance Decision: SCHEDULE INSPECTION
Alert Level: WARNING
--------------------------------------------
1. ASSESSMENT  
The industrial robot ROBOT_007 is currently showing early signs of degradation, specifically related to actuator performance. Although the predicted probability of failure remains very low, and the remaining useful life is estimated to be about 49 cycles, the system has detected a change in state indicating a developing issue.

2. EVIDENCE  
- Failure probability is 0.0002 (0.02%).  
- Predicted remaining useful life is approximately 49.12 cycles.  
- Overall status is set to WARNING with a high maintenance priority.  
- Maintenance decision is to schedule an inspection due to machine degradation.  
- Alert level is WARNING and an alert message has been generated accordingly.  

In [19]:
# ============================================
# STEP 20: SAVE LLM MAINTENANCE REPORTS
# ============================================

from pathlib import Path

output_path = Path("../data/processed/llm_maintenance_reports.csv")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

llm_reports_df.to_csv(
    output_path,
    index=False
)

print("LLM maintenance reports saved successfully.")
print("File:", output_path)
print("Rows:", len(llm_reports_df))
print("Columns:", len(llm_reports_df.columns))


LLM maintenance reports saved successfully.
File: ../data/processed/llm_maintenance_reports.csv
Rows: 55
Columns: 12


In [20]:
# ============================================
# STEP 21: VERIFY SAVED FILE
# ============================================

if not output_path.exists():
    raise FileNotFoundError(
        f"Expected output file was not created: {output_path}"
    )

saved_reports_df = pd.read_csv(output_path)

print("File exists: True")
print(
    "File size:",
    round(output_path.stat().st_size / 1024, 2),
    "KB"
)
print("Reloaded shape:", saved_reports_df.shape)

if len(saved_reports_df) == len(llm_reports_df):
    print("Verification successful: saved row count matches generated row count.")
else:
    print("WARNING: saved row count does not match generated row count.")

display(saved_reports_df.head(3))


File exists: True
File size: 95.05 KB
Reloaded shape: (55, 12)
Verification successful: saved row count matches generated row count.


,Machine_ID,Machine_Type,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_priority,maintenance_decision,alert_level,alert_required,llm_report
0,ROBOT_007,Industrial_Robot,157,ACTUATOR_DEGRADATION,0.0002,49.12,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
1,ROBOT_007,Industrial_Robot,161,ACTUATOR_DEGRADATION,0.0003,48.37,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...
2,ROBOT_007,Industrial_Robot,163,ACTUATOR_DEGRADATION,0.0002,48.41,WARNING,HIGH,SCHEDULE INSPECTION,WARNING,True,1. ASSESSMENT \nThe industrial robot ROBOT_00...


In [21]:
# ============================================
# STEP 22: LLM REPORT SUMMARY
# ============================================

print("============================================")
print("LLM MAINTENANCE REPORT SUMMARY")
print("============================================")

print("Total actionable events:", len(llm_reports_df))
print(
    "Unique machines:",
    llm_reports_df["Machine_ID"].nunique()
)

print("\nMaintenance decisions:")
display(
    llm_reports_df["maintenance_decision"]
    .value_counts()
    .rename_axis("maintenance_decision")
    .reset_index(name="count")
)

print("\nAlert levels:")
display(
    llm_reports_df["alert_level"]
    .value_counts()
    .rename_axis("alert_level")
    .reset_index(name="count")
)

print("\nOverall status:")
display(
    llm_reports_df["overall_status"]
    .value_counts()
    .rename_axis("overall_status")
    .reset_index(name="count")
)

print("\nReports by machine:")
display(
    llm_reports_df["Machine_ID"]
    .value_counts()
    .rename_axis("Machine_ID")
    .reset_index(name="report_count")
)

print("\nFinal LLM artifact:")
print(output_path)


LLM MAINTENANCE REPORT SUMMARY
Total actionable events: 55
Unique machines: 7

Maintenance decisions:


,maintenance_decision,count
0,IMMEDIATE MAINTENANCE,35
1,SCHEDULE INSPECTION,20



Alert levels:


,alert_level,count
0,CRITICAL,35
1,WARNING,20



Overall status:


,overall_status,count
0,CRITICAL,35
1,WARNING,19
2,HEALTHY,1



Reports by machine:


,Machine_ID,report_count
0,ROBOT_007,9
1,MOTOR_004,9
2,COMP_003,9
3,PRESS_006,8
4,CNC_001,7
5,PUMP_002,7
6,TURB_005,6



Final LLM artifact:
../data/processed/llm_maintenance_reports.csv


## Final Output

After successful execution, this notebook creates:

```text
data/processed/llm_maintenance_reports.csv
```

The complete pipeline represented by this notebook is:

```text
Orchestrator
    ↓
1,930 runtime events
    ↓
55 actionable events
    ↓
LLM Maintenance Agent
    ↓
55 maintenance intelligence reports
    ↓
llm_maintenance_reports.csv
```

The LLM is used as an **explanation/intelligence layer**. It does not replace the deterministic Failure Prediction, RUL, Maintenance Agent, Alert Agent, or Orchestrator decisions.
